# A FAIRE POUR LANCER UN GROS ENTRAINEMENT

- utiliser un modèle b3 (ou +) que b0
- en csq changer le target size dans le preprocessor
- augmenter nombre d'epoch
- batch size ?

Optionnel :
- tester une autre loss

# **0. Librairies**

In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import sys
import os




sys.path.append(os.path.abspath(".."))

from lib.data.dataset import BeeDataset

from lib.data.preprocessing import TorchPreprocessor

from lib.data.train_val_split import train_val_split

from lib.data.preprocessing import TorchPreprocessor

from lib.data.data_augmentation import data_augmented_loader

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 8
EPOCHS = 25
LR = 1e-4
num_classes = 50

notebook_dir = os.getcwd()

data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "data"))

## **1. Preprocessing**

In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# preprocessor = TorchPreprocessor(
#     normalize=True,
#     mean = IMAGENET_MEAN,
#     std = IMAGENET_STD,
#     resize_method="pad",
#     target_size=(224, 224),
# )

# train_dataset, val_dataset = train_val_split(train_transform=preprocessor, val_transform=preprocessor)


In [4]:
import lib.data.preprocessing as prep
print(prep.__file__)

/home/alexandre-tonon/SDD/Hackathons/Hackaton_abeilles_tigres/lib/data/preprocessing.py


In [ ]:
heavy_training_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="heavy",
    resize_method="crop",
    resize_value=320,
    target_size=(300, 300),
    interpolation_method="BICUBIC",
)

light_training_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="light",
    resize_method="crop",
    resize_value=320,
    target_size=(300, 300),
    interpolation_method="BICUBIC",
)

val_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="none",
    resize_method="crop",
    resize_value=320,
    target_size=(300, 300),
    interpolation_method="BICUBIC",
)


train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, target_size=(300, 300), batch_size=BATCH_SIZE, train_preprocessor_light=light_training_preprocessor, train_preprocessor_heavy=heavy_training_preprocessor, val_preprocessor=val_preprocessor)

Train prêt : 6417 images (avec augmentation ciblée)
Val prête  : 1582 images (sans augmentation)


## **2. Modèle**

In [6]:
from torch.optim.lr_scheduler import CosineAnnealingLR

def create_model(num_classes: int) -> nn.Module:
    model = models.efficientnet_b3(weights="IMAGENET1K_V1") #mettre b3 si ca marche
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes),
    )
    return model

model = create_model(num_classes)
model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

# --- Variante ---
# pip install torchmetrics
# from torchmetrics.classification import MulticlassFocalLoss
# criterion = MulticlassFocalLoss(num_classes=num_classes, alpha=0.25, gamma=2.0)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), #ajout par rapport au code précédent
    lr=LR,
    weight_decay=1e-4
)

scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

## **3. F1-score**

In [7]:
import numpy as np

def compute_f1(all_labels, all_preds, num_classes):
    f1_per_class = []

    for cls in range(num_classes):
        # True Positives
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        # False Positives
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        # False Negatives
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    # F1 macro : moyenne des classes
    f1_macro = np.mean(f1_per_class)
    return f1_macro, f1_per_class

## **4. Fonctions de training et validation**

In [8]:
def train_one_epoch():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # 🔹 Calcul F1 avec ta fonction existante
    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, num_classes)

    # 🔹 Affichage
    # print(f"Train F1 macro: {f1_macro:.4f}")

    return total_loss / len(train_loader), correct / total, f1_macro, f1_per_class


In [9]:
import torch

def validate():
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calcul F1 manuel par classe
    f1_per_class = []
    for cls in range(num_classes):
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))
        
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
        
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)
    
    f1_macro = np.mean(f1_per_class)

    return total_loss / len(val_loader), f1_macro, f1_per_class

## **5. Entrainement**

**Vérif des labels**

In [10]:
# all_labels = [label for _, label in train_dataset]
# print("Label min:", min(all_labels))
# print("Label max:", max(all_labels))
# print("Nombre de classes uniques:", len(set(all_labels)))

# # Récupérer tous les labels uniques triés
# all_labels = sorted(set(label for _, label in train_dataset.samples))
# label_to_index = {label: i for i, label in enumerate(all_labels)}

# # Remapper les labels dans le dataset
# # for i, (path, label) in enumerate(train_dataset.samples):
# #     train_dataset.samples[i] = (path, label_to_index[label])

**Entrainement**

In [11]:
import csv
best_f1 = 0.0
best_model_path = "best_model.pth"

# Configuration du logging CSV
csv_file = "training_log.csv"
fieldnames = ['epoch', 'train_loss', 'train_acc', 'train_f1_macro', 'val_loss', 'val_f1_macro']

# Initialisation du fichier (écrase le précédent s'il existe)
with open(csv_file, mode='w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch()
    scheduler.step()
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Train F1 Macro: {train_f1_macro:.4f}")
    print(f"Train F1 per class: {train_f1_per_class}")

    val_loss, val_f1_macro, val_f1_per_class = validate()
    print(f"Val   Loss: {val_loss:.4f}")
    print(f"Val   F1 Macro: {val_f1_macro:.4f}")
    print(f"Val   F1 per class: {val_f1_per_class}")

    with open(csv_file, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'train_f1_macro': train_f1_macro,
            'val_loss': val_loss,
            'val_f1_macro': val_f1_macro
        })

    # 🔹 Sauvegarde du meilleur modèle
    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro
        torch.save(model.state_dict(), best_model_path)
        print(f" Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")

100%|██████████| 803/803 [07:27<00:00,  1.80it/s]



Epoch 1/25
Train Loss: 2.5393 | Train Acc: 0.4256
Train F1 Macro: 0.4197
Train F1 per class: [np.float64(0.6666666666666666), np.float64(0.7253521126760565), np.float64(0.11707317073170731), np.float64(0.5228215767634855), np.float64(0.3012048192771084), np.float64(0.23699421965317918), np.float64(0.26519337016574585), np.float64(0.5662100456621005), np.float64(0.7773584905660378), np.float64(0.48000000000000004), np.float64(0.7032967032967032), np.float64(0.4134078212290503), np.float64(0.14583333333333334), np.float64(0.5523012552301254), np.float64(0.25), np.float64(0.18181818181818182), np.float64(0.425531914893617), np.float64(0.2012578616352201), np.float64(0.3393501805054152), np.float64(0.46913580246913583), np.float64(0.6996699669966996), np.float64(0.089171974522293), np.float64(0.3938461538461539), np.float64(0.5463414634146341), np.float64(0.1503267973856209), np.float64(0.33497536945812806), np.float64(0.7012987012987012), np.float64(0.6115107913669064), np.float64(0.1179

100%|██████████| 803/803 [08:25<00:00,  1.59it/s]



Epoch 2/25
Train Loss: 0.9549 | Train Acc: 0.7639
Train F1 Macro: 0.7538
Train F1 per class: [np.float64(0.9734513274336283), np.float64(0.9641434262948209), np.float64(0.29556650246305416), np.float64(0.9), np.float64(0.8550185873605948), np.float64(0.6666666666666666), np.float64(0.8072727272727274), np.float64(0.9489051094890512), np.float64(0.9618320610687022), np.float64(0.8028673835125447), np.float64(0.9694323144104805), np.float64(0.7872340425531915), np.float64(0.4031620553359684), np.float64(0.9220338983050848), np.float64(0.5420560747663552), np.float64(0.6891385767790261), np.float64(0.7942583732057416), np.float64(0.39195979899497485), np.float64(0.6056338028169014), np.float64(0.8273092369477911), np.float64(0.9672727272727273), np.float64(0.625), np.float64(0.6343042071197412), np.float64(0.7908745247148289), np.float64(0.4102564102564103), np.float64(0.6204379562043796), np.float64(0.9632352941176471), np.float64(0.9351535836177475), np.float64(0.45132743362831856), np

100%|██████████| 803/803 [08:45<00:00,  1.53it/s]


...


Epoch 4/25
Train Loss: 0.4319 | Train Acc: 0.8825
Train F1 Macro: 0.8816
Train F1 per class: [np.float64(0.99), np.float64(1.0), np.float64(0.6206896551724138), np.float64(0.9711934156378601), np.float64(0.983739837398374), np.float64(0.7628205128205129), np.float64(0.9527272727272726), np.float64(0.9803921568627452), np.float64(0.9915966386554622), np.float64(0.9328063241106719), np.float64(0.9895470383275262), np.float64(0.9469964664310955), np.float64(0.6928571428571428), np.float64(0.979253112033195), np.float64(0.8540145985401459), np.float64(0.944954128440367), np.float64(0.9647058823529412), np.float64(0.7601809954751132), np.float64(0.7862595419847329), np.float64(0.8352490421455939), np.float64(0.9881422924901185), np.float64(0.937984496124031), np.float64(0.753623188405797), np.float64(0.9420849420849421), np.float64(0.652014652014652), np.float64(0.7614678899082569), np.float64(0.9802371541501976), np.float64(0.9879518072289156), np.float64(0.6693227091633466), np.float64(0

100%|██████████| 803/803 [08:35<00:00,  1.56it/s]



Epoch 5/25
Train Loss: 0.3499 | Train Acc: 0.9009
Train F1 Macro: 0.9008
Train F1 per class: [np.float64(0.9959183673469388), np.float64(0.9962546816479401), np.float64(0.6396761133603239), np.float64(0.9547325102880658), np.float64(0.9777777777777777), np.float64(0.7372262773722628), np.float64(0.967741935483871), np.float64(0.9925925925925926), np.float64(0.9923664122137404), np.float64(0.9776119402985075), np.float64(1.0), np.float64(0.945736434108527), np.float64(0.6857142857142857), np.float64(0.9714285714285714), np.float64(0.9224806201550388), np.float64(0.9571428571428572), np.float64(0.9791666666666666), np.float64(0.8174603174603176), np.float64(0.8321167883211679), np.float64(0.9185185185185185), np.float64(1.0), np.float64(0.929368029739777), np.float64(0.8333333333333335), np.float64(0.9365079365079365), np.float64(0.6907630522088354), np.float64(0.7958477508650519), np.float64(0.9959183673469388), np.float64(0.99581589958159), np.float64(0.7074235807860262), np.float64(0

  3%|▎         | 27/803 [00:18<08:51,  1.46it/s]


KeyboardInterrupt: 

## **6. Création du fichier submission**

In [ ]:
from torch.utils.data import DataLoader
import pandas as pd
import torch
from lib.data.dataset import BeeDataset

def submission(model, batch_size=32, transform=None, model_path="best_model.pth", save_path="submission.csv"):
    # Charger le modèle sur le bon device
    model.load_state_dict(torch.load(model_path, map_location=torch.device(DEVICE)))
    model.to(DEVICE)
    model.eval()
    
    # Dataset de test
    test_dataset = BeeDataset(train=False, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    all_ids = []
    all_preds = []
    
    with torch.no_grad():
        for imgs, ids in test_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            
            preds = torch.argmax(outputs, dim=1)
            
            # Convertir preds en int et ids en int ou str
            all_preds.extend(preds.cpu().tolist())
            all_ids.extend([int(x) if isinstance(x, torch.Tensor) else x for x in ids])
    
    submission_df = pd.DataFrame({
        "id": all_ids,
        "label": all_preds
    })
    
    submission_df.to_csv(save_path, index=False)
    print(f"Submission saved to {save_path}")

In [ ]:
preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    resize_method="pad",
    target_size=(224, 224),
)


test_dataset = BeeDataset(train=False, transform=preprocessor)


submission(model, batch_size=32, transform=preprocessor, save_path="submission.csv")